In [1]:
import matplotlib.pyplot as plt
from time import time
import numpy as np
import os
import cv2
import json
import torch
from timm.models.vision_transformer import VisionTransformer, PatchEmbed
import torch
import torch.nn as nn

import numpy as np
import os
import re
import timm

In [2]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
img_folder = "/home/viraj/my_project/val/val2/"
output_folder = "reconstred_line_0_circle20_thres_.4_pixel50_n12_vit_base_VAL_DATASET"
txt_path="similarity_score_with_cls_token_accuracy_drop_10_percent.txt"
json_file = "/home/viraj/my_project/imagenet_class_index.json"
file_path2='average_percentag_0_4_55_87_cr_18.txt'
black_intensity=0.0
circle_r=18
# line_r=0
average_percent_dro_embedding=10
drop_rate=0.10
drop_rate_layers=0.00

In [3]:
# pretrained_vit = vit_large_patch16_224(pretrained=True).to(device)

pretrained_vit = timm.create_model('vit_base_patch16_224', pretrained=True).to(device)

In [4]:
def append_to_txt(file_path, keypoints_value):
    with open(file_path, 'a') as file:
        file.write(f'keypoints_value: {keypoints_value}\n')

In [5]:
scan=[]
keypoints_counts_every_time=[]
def find_keypoints(gray_image):
           #np.mean(gray_image)/2
    count = 0

    THRESHOLDS = 0.30
    scan_count=0

    while count <= 100 and THRESHOLDS > 0:
        scale_kp1 = FAST(gray_image, N=12, threshold=THRESHOLDS, nms_window=3)
        y = len(scale_kp1)

        count = y
        
        print(f"Threshold: {THRESHOLDS}, Keypoints Count: {count}")
        THRESHOLDS= THRESHOLDS*0.5
        # append_to_txt('keypoints_values_t0.30_count100_bi_79.txt', y)
        keypoints_counts_every_time.append(y)
        scan_count+=1
    scan.append(scan_count)
    # append_to_txt('scan_count_values_no_500.txt', scan_count)
    return np.array(scale_kp1)


In [6]:
import numpy as np

def draw_keypoints(scale_kp1, gray1, circle_radius=circle_r):
    """
    Draw keypoints on the image as circles.

    Args:
    - scale_kp1: List of keypoints, where each keypoint is a tuple (x, y)
    - gray1: Grayscale image (2D list or array)
    - circle_radius: Radius of the circle (default is 5)

    Returns:
    - features_img1: Image with keypoints drawn as circles
    """
    features_img1 = np.array(gray1)
    # print(features_img1)
    features_img1=features_img1.astype(np.int16)
    # print(features_img1)
 
    for keypoint in scale_kp1:
        x, y = int(keypoint[0]), int(keypoint[1])
        

        # Ensure circle boundaries stay within the image dimensions
        x_min = max(0, x - circle_radius)
        x_max = min(features_img1.shape[1] - 1, x + circle_radius)
        y_min = max(0, y - circle_radius)
        y_max = min(features_img1.shape[0] - 1, y + circle_radius)

        # Draw filled circle
        for i in range(x_min, x_max + 1):
            for j in range(y_min, y_max + 1):
                if (i - x) ** 2 + (j - y) ** 2 <= circle_radius ** 2:
                    features_img1[j, i] = -1  # Set pixel value to -1
    # np.set_printoptions(threshold=np.inf)
    # print(features_img1)

    return features_img1


In [7]:
def plot(image):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.show()

In [8]:
def canva(features_img1):                
    # Parameters
    patch_size = 16

    # Get image dimensions
    height, width = features_img1.shape

    # Calculate the number of patches along each dimension
    num_patches_y = height // patch_size
    num_patches_x = width // patch_size
    mask=[]

    # Process each patch
    for i in range(num_patches_y):
        for j in range(num_patches_x):
            # Calculate the coordinates of the current patch
            y_start = i * patch_size
            y_end = y_start + patch_size
            x_start = j * patch_size
            x_end = x_start + patch_size

            # Extract the patch
            patch = features_img1[y_start:y_end, x_start:x_end]
            # np.set_printoptions(threshold=np.inf)
            # print(patch)
            total_pixels = patch.size
            black_pixels = np.count_nonzero(patch == -1)
            black_pixel_percentage = (black_pixels / total_pixels) * 100

            # Calculate the average intensity of the patch
            
            
            # avg_intensity = np.mean(patch)
            if black_pixel_percentage >= black_intensity :
                mask.append(1)
            else:
                mask.append(0)
                
    mask.insert(0,1)                
    return  mask

In [9]:
def copy_image(img):
    """
    Create a copy of the input image without using any libraries.

    Args:
    - img: Input image (list of lists representing pixel values)

    Returns:
    - img_copy: A copy of the input image
    """
    # Determine the dimensions of the image
    height = len(img)
    width = len(img[0])

    # Create an empty list to store the copied image
    img_copy = []

    # Iterate over each row of the image
    for i in range(height):
        # Create a copy of the current row
        row_copy = img[i][:]  # or row_copy = list(img[i]) for deep copy
        # Append the copied row to the copied image
        img_copy.append(row_copy)

    return img_copy

In [10]:
def bgr_to_rgb(img):
    """
    Convert an image from BGR to RGB color space.

    Args:
    - img: Input image in BGR format (numpy array)

    Returns:
    - img_rgb: Image converted to RGB color space
    """
    # Reverse the order of color channels
    img_rgb = img[:, :, ::-1]

    return img_rgb

In [11]:

def conv2d(img, kernel):
    """
    Perform a 2D convolution on an image using the provided kernel.

    Parameters:
    img (numpy.ndarray): Input image, a 2D array.
    kernel (numpy.ndarray): Convolution kernel, a 2D array.

    Returns:
    numpy.ndarray: The result of the convolution, with the same size as the input image.
    """
    # Flip the kernel (needed for convolution)
    kernel = np.flipud(np.fliplr(kernel))
    
    # Determine the padding size
    pad_h = kernel.shape[0] // 2
    pad_w = kernel.shape[1] // 2
    
    # Pad the image with zeros on all sides
    padded_img = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)
    
    # Initialize the output image
    output = np.zeros_like(img)
    
    # Perform the convolution
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            # Extract the region of interest
            region = padded_img[i:i+kernel.shape[0], j:j+kernel.shape[1]]
            # Perform element-wise multiplication and sum the result
            output[i, j] = np.sum(region * kernel)
    
    return output

In [12]:
def FAST(img, N=9, threshold=0.15, nms_window=0):
    kernel = np.array([[1, 2, 1],
                       [2, 4, 2],
                       [1, 2, 1]]) / 16  # 3x3 Gaussian Window

    img = conv2d(img, kernel)

    cross_idx = np.array([[3, 0, -3, 0], [0, 3, 0, -3]])
    circle_idx = np.array([[3, 3, 2, 1, 0, -1, -2, -3, -3, -3, -2, -1, 0, 1, 2, 3],
                           [0, 1, 2, 3, 3, 3, 2, 1, 0, -1, -2, -3, -3, -3, -2, -1]])

    corner_img = np.zeros(img.shape)
    keypoints = []
    for y in range(3, img.shape[0] - 3,1):
        for x in range(3, img.shape[1] - 3,1):
            Ip = img[y, x]
            
            t = threshold * Ip if threshold < 1 else threshold

            # Fast checking using cross pattern
            if (
                    np.count_nonzero(Ip + t < img[y + cross_idx[0, :], x + cross_idx[1, :]]) >= 3
                    or np.count_nonzero(Ip - t > img[y + cross_idx[0, :], x + cross_idx[1, :]]) >= 3
            ):
                # Keypoint found in cross pattern
                keypoints.append([x, y])  # Note: keypoint = [col, row]
                corner_img[y, x] = np.sum(np.abs(Ip - img[y + circle_idx[0, :], x + circle_idx[1, :]]))

            else:
                # No keypoint found in cross pattern, perform detailed circular checking
                if (
                        np.count_nonzero(img[y + circle_idx[0, :], x + circle_idx[1, :]] >= Ip + t) >= N
                        or np.count_nonzero(img[y + circle_idx[0, :], x + circle_idx[1, :]] <= Ip - t) >= N
                ):
                    # Keypoint found in circular pattern
                    keypoints.append([x, y])  # Note: keypoint = [col, row]
                    corner_img[y, x] = np.sum(np.abs(Ip - img[y + circle_idx[0, :], x + circle_idx[1, :]]))

    # Non-maximal Suppression (NMS) if enabled
    if nms_window != 0:
        fewer_kps = []
        for [x, y] in keypoints:
            window = corner_img[y - nms_window:y + nms_window + 1, x - nms_window:x + nms_window + 1]
            loc_y_x = np.unravel_index(window.argmax(), window.shape)
            x_new = x + loc_y_x[1] - nms_window
            y_new = y + loc_y_x[0] - nms_window
            new_kp = [x_new, y_new]
            if new_kp not in fewer_kps:
                fewer_kps.append(new_kp)
    else:
        fewer_kps = keypoints

    return np.array(fewer_kps)



In [13]:
def draw_lines(image, points, color=0, thickness=24):
    """
    Draw lines between consecutive points on an image.

    Parameters:
    - image: The image (NumPy array) to draw on.
    - points: A list of points (each point is a tuple of coordinates).
    - color: The color of the lines (default is black).
    - thickness: The thickness of the lines (default is 20).
    
    """
    for i in range(len(points) - 1):
        pt1 = tuple(int(coord) for coord in points[i])
        pt2 = tuple(int(coord) for coord in points[i + 1])
        draw_line(image, pt1, pt2, color, thickness)

In [14]:

def draw_line(image, pt1, pt2, color=0, thickness=10):
    """
    Draw a line on the image using Bresenham's line algorithm.

    Parameters:
    - image: The image (NumPy array) to draw on.
    - pt1: The starting point of the line.
    - pt2: The ending point of the line.
    - color: The color of the line (default is black).
    - thickness: The thickness of the line (default is 20).
    """
    x1, y1 = pt1
    x2, y2 = pt2
    dx = abs(x2 - x1)
    dy = abs(y2 - y1)
    sx = 1 if x1 < x2 else -1
    sy = 1 if y1 < y2 else -1
    err = dx - dy

    while True:
        for i in range(-thickness // 2, thickness // 2 + 1):
            for j in range(-thickness // 2, thickness // 2 + 1):
                if 0 <= x1 + i < image.shape[1] and 0 <= y1 + j < image.shape[0]:
                    image[y1 + j, x1 + i] = color
        if x1 == x2 and y1 == y2:
            break
        e2 = err * 2
        if e2 > -dy:
            err -= dy
            x1 += sx
        if e2 < dx:
            err += dx
            y1 += sy

In [15]:
def resize_nearest_neighbor(image, size):
    original_height = len(image)
    original_width = len(image[0])
    new_width, new_height = size

    if isinstance(image[0][0], list) or isinstance(image[0][0], np.ndarray):
        # For color images
        resized_image = np.zeros((new_height, new_width, 3), dtype=np.uint8)
        for i in range(new_height):
            for j in range(new_width):
                # Calculate the nearest pixel in the original image
                x = int(j * original_width / new_width)
                y = int(i * original_height / new_height)
                resized_image[i, j] = image[y][x]
    else:
        # For grayscale images
        resized_image = np.zeros((new_height, new_width), dtype=np.uint8)
        for i in range(new_height):
            for j in range(new_width):
                # Calculate the nearest pixel in the original image
                x = int(j * original_width / new_width)
                y = int(i * original_height / new_height)
                resized_image[i, j] = image[y][x]
    
    return np.array(resized_image)

In [16]:
def save_average_percentage(file_path, average_percent, accuracy, processed_image, w):
    with open(file_path, 'a') as file:
        file.write(f"Epoch: Average percentage_patch_drop: {average_percent}, Accuracy: {accuracy}, processed_image: {processed_image}, scan: {w}\n")

In [17]:
def load_mapping_file(json_file):
    with open(json_file, 'r') as f:
        mapping_data = json.load(f)
    return mapping_data

In [18]:
import torchvision.transforms as transforms
def transform_image(image, device):

    image=transforms.ToTensor()(image)
    image= image.unsqueeze(0)
    image=image.to(device)    
    

    
    return image

In [19]:
mapping_data = load_mapping_file(json_file)

def get_key_by_value(predicted_index, class_name):
    
    for key, values in mapping_data.items():
        key = int(key)
        
        if class_name in values:
            # print(key, predicted_index, "key and predicted_index")
            # print(f"Found class_name '{class_name}' in values: {values}")
            if int(key) == predicted_index:
                return True
    return False

In [20]:
import numpy as np

def rgb_to_gray(image):
    """
    Convert an RGB image to grayscale.

    Parameters:
    image (numpy.ndarray): Input RGB image as a NumPy array.

    # Returns:
    numpy.ndarray: Grayscale image.
    """
    if len(image.shape) != 3 or image.shape[2] != 3:
        raise ValueError("Input image must be an RGB image with 3 channels.")
    
    # Extract the R, G, B channels
    R = image[:, :, 0]
    G = image[:, :, 1]
    B = image[:, :, 2]
    
    # Apply the grayscale conversion formula
    gray = 0.299 * R + 0.587 * G + 0.114 * B
    
    # Convert to uint8 (0-255 range)
    gray = np.clip(gray, 0, 255).astype(np.uint8)
    
    return gray

In [21]:
patch_drop_percentage = []
def average_patch_drop(mask):
    count_ones = mask.count(1)
    count_ones=count_ones-1
    y = (((196 - count_ones ) / 196) * 100)
    patch_drop_percentage.append(y)
    

In [22]:
import torch
import torch.nn as nn
import timm
from timm.models.vision_transformer import VisionTransformer, Block
import numpy as np
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt

# count=0
class CustomAttention(nn.Module):
    def __init__(self, attn, num_heads):
        super(CustomAttention, self).__init__()
        self.num_heads = num_heads
        self.scale = attn.scale
        self.qkv = attn.qkv
        self.attn_drop = attn.attn_drop
        self.proj = attn.proj
        self.proj_drop = attn.proj_drop

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        mask_attention = None
        avg_attn = attn.mean(dim=1)
        cls_token_attn = avg_attn[:, 0, :] 
        threshold = torch.quantile(cls_token_attn, 0.00, dim=-1, keepdim=True)
        mask_attention = cls_token_attn >= threshold
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x, mask_attention


# class PatchDropout(nn.Module):
#     def __init__(self):
#         super().__init__()

#     def forward(self, x, mask):
#         mask_tensor = torch.tensor(mask, dtype=torch.bool)
#         indices = torch.nonzero(mask_tensor).squeeze() 
#         tensor2 = x[:, indices, :]
#         print(tensor2.shape, "tensor2")
#         return tensor2


class PatchDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.count = 0
        # self.drop_rate = drop_rate  # Drop 20% of the tokens

    def forward(self, x, mask):


        batch_size, seq_len, embedding_dim = x.shape
        assert seq_len == 197, "Expected sequence length of 197, but got {}".format(seq_len)

        # Extract CLS token (first token)
        cls_token = x[:, 0, :]  # Shape: (batch_size, embedding_dim)

        # Compute cosine similarity with all other tokens
        similarity_scores = F.cosine_similarity(x[:, 1:, :], cls_token.unsqueeze(1), dim=-1)  # (batch_size, 196)

# Ensure the similarity scores are reshaped into a 16x16 matrix
        similarity_matrix = similarity_scores.detach().cpu().numpy().reshape(14, 14)

# Plot the heatmap
        fig, ax = plt.subplots(figsize=(10, 10))
        sns.heatmap(similarity_matrix, cmap="viridis", square=True, annot=False, cbar=True, ax=ax)

        plt.title("16x16 Patch Similarity Heatmap")
        plt.xlabel("Patch Index (X)")
        plt.ylabel("Patch Index (Y)")
        plt.show()
        # print(similarity_scores.shape)
        sorted_scores=(torch.sort(similarity_scores))
        mean_similarity=(torch.mean(similarity_scores))
        # print("mean_scores", mean_similarity)
        # print(sorted_scores)
        greater_equal_mask = similarity_scores >= mean_similarity
        less_mask = similarity_scores < mean_similarity
        print(less_mask)
        print(greater_equal_mask)
        
        
        greater_equal_tokens = x[:, 1:][greater_equal_mask]  # Tokens with similarity >= mean
        less_tokens = x[:, 1:][less_mask]  # Tokens with similarity < mean
        print("Greater-equal tokens shape:", greater_equal_tokens.shape)
        print("Less tokens shape:", less_tokens.shape)

        # low_similarity_threshold = 0.85

        # low_similarity_tokens = []
        # for token in greater_equal_tokens:
        #     similarities = F.cosine_similarity(token.unsqueeze(0), less_tokens, dim=-1)
        #     sorted_simi= torch.sort(similarities, dim= -1)
        #     # print("sorted_similarities_scores", sorted_simi)
        #     if torch.all(similarities < low_similarity_threshold):
        #         # print("1")
        #         continue
        #     else:low_similarity_tokens.append(token)





        # if low_similarity_tokens:
        #     low_similarity_tokens = torch.stack(low_similarity_tokens)
        #     less_tokens = torch.cat([less_tokens, low_similarity_tokens], dim=0)
        #     print("Low similarity tokens shape:", low_similarity_tokens.shape)
        # else:
        #     print("No low-similarity tokens found.")
        
   
        # print("Final less_tokens shape (after appending):", less_tokens.shape)

        less_tokens = less_tokens.unsqueeze(0)
        cls_token_expanded = cls_token.unsqueeze(1)
        less_tokens_with_cls = torch.cat([less_tokens, cls_token_expanded], dim=1)
        length= less_tokens_with_cls.shape
        print("Final less_tokens shape (after appending):", length)

        
        # # count+=length
        # # ✅ Append the length to a text file
        # with open("token_lengths.txt", "a") as f:
        #     f.write(f"{length}\n")

        
         
        # less_tokens_new =torch.cat([less_tokens, cls_token_expanded], dim=1)


        

        
        # # Determine threshold to drop top 20% most similar tokens
        # k = int(seq_len * drop_rate)  # 20% of 196 tokens (~39 tokens)
        # threshold_values, _ = torch.kthvalue(similarity_scores, seq_len - k-1 , dim=1, keepdim=True)  # (batch_size, 1)
        # # print(threshold_values)

        # # Create a mask to **remove top 20% (high similarity scores)**
        # keep_mask = similarity_scores <= threshold_values  # (batch_size, 196)
        
        # # Append True for CLS token (always keep it)
        # keep_mask = torch.cat([torch.ones(batch_size, 1, dtype=torch.bool, device=x.device), keep_mask], dim=1)  # (batch_size, 197)

        # Apply mask and return filtered tensor
        # x_filtered = x[keep_mask].view(batch_size, -1, embedding_dim)  # Adjust shape after filtering
        # print(x_filtered.shape)
        return  less_tokens_with_cls

class CustomBlock(nn.Module):
    def __init__(self, block):
        super(CustomBlock, self).__init__()
        self.norm1 = block.norm1
        self.attn = CustomAttention(block.attn, block.attn.num_heads)
        self.ls1 = block.ls1
        self.drop_path1 = block.drop_path1
        self.norm2 = block.norm2
        self.mlp = block.mlp
        self.ls2 = block.ls2
        self.drop_path2 = block.drop_path2

    def forward(self, x, apply_patch_drop=False):

        
        x_residual = self.norm1(x)
        x_residual, mask_attention = self.attn(x_residual)
        x = x + self.drop_path1(self.ls1(x_residual))
        x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))
        if apply_patch_drop:
            
            
            batch_size, seq_len, embedding_dim = x.shape
            # assert seq_len == 197, "Expected sequence length of 197, but got {}".format(seq_len)
    
            # Extract CLS token (first token)
            cls_token = x[:, 0, :]  # Shape: (batch_size, embedding_dim)
    
            # Compute cosine similarity with all other tokens
            similarity_scores = F.cosine_similarity(x[:, 1:, :], cls_token.unsqueeze(1), dim=-1)  # (batch_size, 196)
            # Determine threshold to drop top 20% most similar tokens
            k = int(seq_len *  drop_rate_layers)  # 20% of 196 tokens (~39 tokens)
            # print(k, "k_value")
            threshold_values, _ = torch.kthvalue(similarity_scores, seq_len - k-1 , dim=1, keepdim=True)  # (batch_size, 1)
            # print(threshold_values,"thresholdssss")
    
            # Create a mask to **remove top 20% (high similarity scores)**
            keep_mask = similarity_scores <= threshold_values  # (batch_size, 196)
            # print(keep_mask.shape, "keep_mask")
            
            # Append True for CLS token (always keep it)
            keep_mask = torch.cat([torch.ones(batch_size, 1, dtype=torch.bool, device=x.device), keep_mask], dim=1)  # (batch_size, 197)
    
            # Apply mask and return filtered tensor
            x_filtered = x[keep_mask].view(batch_size, -1, embedding_dim)  # Adjust shape after filtering
            # print(x_filtered.shape)
            return x_filtered
            
        return x

class CustomVisionTransformer(VisionTransformer):
    def __init__(self, patch_drop_blocks=None, **kwargs):
        super(CustomVisionTransformer, self).__init__(**kwargs)
        self.patch_drop_blocks = patch_drop_blocks if patch_drop_blocks is not None else []
        self.patch_drop = PatchDropout()
        self.blocks = nn.Sequential(*[
            CustomBlock(block) if i in self.patch_drop_blocks else block
            for i, block in enumerate(self.blocks)
        ])

    def forward(self, x, mask=None):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # class token
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        if mask is not None:
            x = self.patch_drop(x, mask)

        for i, block in enumerate(self.blocks):
            if isinstance(block, CustomBlock):
                x = block(x, apply_patch_drop=True)
                # print(i, x.shape)
            else:
                x = block(x)
        x = self.norm(x)
        x = self.head(x[:, 0])
        return x

# Example usage
my_array = np.random.randint(1, 2, size=197)  # Updated to have values 0 or 1
custom_vit = CustomVisionTransformer(
    img_size=224, patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True,
    norm_layer=nn.LayerNorm, num_classes=1000, patch_drop_blocks=[]
).to(device)


# Load pretrained weights
# pretrained_vit = timm.create_model('vit_base_patch16_224', pretrained=True)
custom_vit.load_state_dict(pretrained_vit.state_dict(), strict=True)

# Forward pass with input tensor and masks
input_tensor = torch.randn(1, 3, 224, 224).to(device)
output = custom_vit(input_tensor, mask=my_array)
print(output.shape)  # Should be [1, 1000]


<Figure size 1000x1000 with 2 Axes>

tensor([[False, False, False,  True, False, False,  True,  True, False, False,
         False, False, False,  True, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
          True, False,  True, False, False,  True, False,  True, False, False,
          True, False,  True, False,  True,  True,  True,  True,  True,  True,
         False, False, False, False,  True, False, False, False, False,  True,
          True,  True,  True,  True, False,  True,  True, False, False, False,
          True, False,  True,  True,  True,  True,  True,  True,  True,  True,
         False, False, False, False,  True, False,  True,  True,  True,  True,
          True,  True,  True, False, False,  True, False, False, False,  True,
          True,  True,  True,  True,  True,  True,  True, False, False,  True,
         False, False,  True, False, False, False,  True, False,  True,  True,
         False, False, False, False, False, False, F

In [23]:
from qtorch.quant import Quantizer, quantizer
from qtorch import FixedPoint
# Define the fixed-point quantizer
weight_quantizer = quantizer(forward_number=FixedPoint(16, 8), forward_rounding="nearest")
act_quantizer = quantizer(forward_number=FixedPoint(16, 8), forward_rounding="nearest")

/usr/local/lib/python3.10/dist-packages/torch/utils/cpp_extension.py:1967: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


In [24]:
import torch
import torch.nn as nn

# Define the QuantizedVisionTransformer class
class QuantizedVisionTransformer(nn.Module):
    def __init__(self, model, weight_quantizer, act_quantizer):
        super(QuantizedVisionTransformer, self).__init__()
        self.model = model
        self.weight_quantizer = weight_quantizer
        self.act_quantizer = act_quantizer
        self.apply(self._quantize_weights)

    def _quantize_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data = self.weight_quantizer(module.weight.data)
            if module.bias is not None:
                module.bias.data = self.weight_quantizer(module.bias.data)
        elif isinstance(module, nn.Conv2d):
            module.weight.data = self.weight_quantizer(module.weight.data)
            if module.bias is not None:
                module.bias.data = self.weight_quantizer(module.bias.data)

    def forward(self, x, mask):
        x = self.act_quantizer(x)
        # for name, module in self.model.named_children():
        #     x = module(x, mask) if 'block' in name else module(x)
        #     if isinstance(module, (nn.Linear, nn.Conv2d)):
        #         x = self.act_quantizer(x)
        return self.model(x, mask)

# Example usage:
# Assume `vision_transformer_model` is your pretrained model
# and `weight_quantizer` and `act_quantizer` are your quantization functions
quantized_model = QuantizedVisionTransformer(custom_vit, weight_quantizer, act_quantizer)



In [25]:
def check_quantization(model):
    for name, param in model.named_parameters():
        print(f"Parameter name: {name}")
        print(f"Data type: {param.dtype}")
        print(f"Values: {param.view(-1)[:10]}")  # Print the first 10 values
        print(f"Storage size (bytes): {param.numel() * param.element_size()}")
        print()



# Check quantized weights and biases
# check_quantization(quantized_model)

In [26]:
output = quantized_model(input_tensor, mask=my_array)

<Figure size 1000x1000 with 2 Axes>

tensor([[False, False, False,  True, False, False,  True,  True, False, False,
         False, False, False,  True, False, False, False, False, False, False,
         False, False, False, False,  True, False, False, False, False, False,
          True, False,  True, False, False, False, False,  True, False, False,
          True, False,  True, False,  True,  True,  True,  True,  True,  True,
         False, False, False, False,  True, False, False, False,  True,  True,
          True,  True,  True,  True,  True,  True,  True, False, False, False,
          True, False,  True,  True,  True,  True,  True,  True,  True,  True,
         False, False, False, False,  True, False,  True,  True,  True,  True,
          True,  True, False, False, False, False, False, False, False,  True,
          True,  True,  True, False,  True,  True,  True, False, False,  True,
         False, False,  True, False, False, False,  True, False,  True,  True,
         False, False, False, False, False, False, F

In [27]:
def main_processing(image):
    resized_image =  resize_nearest_neighbor(image, (224, 224))
    resized_image = bgr_to_rgb(resized_image)
    # gray_image = rgb_to_gray(resized_image)
    resized_image = resized_image.copy()                                        
    image_tensor= transform_image(resized_image, device)
    plot(resized_image)
    # keypoints = find_keypoints(gray_image)  
    # feature_image = draw_keypoints(keypoints, gray_image)
    # mask  = canva(feature_image)
    # average_patch_drop(mask)
    mask = np.random.randint(1, 2, size=197) 
    
    return image_tensor, mask


In [28]:
def save_percentage(file_path, average_percent1, average_percent2, average_percent3 ,average_percent4, average, count):
    with open(file_path, 'a') as file:
        file.write(f"Epoch: Average percentage_patch_drop: { average_percent1}, { average_percent2}, { average_percent3}, { average_percent4}, average: {average}, count{count}\n")

In [29]:
def convert_to_canvas(img_folder, model, device):
    correct_predictions = 0
    total_images = 0
    processed_images = 0
    
    for subfolder_name in os.listdir(img_folder):
        subfolder_path = os.path.join(img_folder, subfolder_name)
        if os.path.isdir(subfolder_path):
            # Process each image in the subfolder
            for img_name in os.listdir(subfolder_path):
                img_path = os.path.join(subfolder_path, img_name)
                if os.path.isfile(img_path) and img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    # Load the image
                    image = cv2.imread(img_path)
                    if image is None:
                        print(f"Failed to load image: {img_path}")
                        continue

                    image_tensor, fast_mask = main_processing(image)
                    
                    # outputs = custom_vit( image_tensor, mask= fast_mask)
                    outputs= quantized_model(image_tensor, mask= fast_mask)
                    predicted_idx = torch.argmax(outputs, dim=1).item()
                    match = re.search(r'val2/([^/]+)/', img_path)
                    if match:
                        class_name = match.group(1)
                    else:
                        class_name = None
                    
                    prediction_result = get_key_by_value(predicted_idx, class_name)
                    
                    if prediction_result is True:
                        correct_predictions += 1    
                    processed_images += 1
                    
                    # Print accuracy after processing every 50 images
                    if processed_images % 10 == 0:
                        # w= ( sum(scan)/ len(scan))
                        w=0
                        accuracy_so_far = (correct_predictions / processed_images) * 100
                        # average_percent = sum(patch_drop_percentage) / len(patch_drop_percentage)
                        
                        # print("average percentage embedding_drop " , average_percent , len(patch_drop_percentage))
                        print(f"Accuracy after processing {processed_images} images: {accuracy_so_far:.2f}%")
                        save_average_percentage(txt_path,  average_percent_dro_embedding, accuracy_so_far, processed_images, w )
                    
                    
                    


In [30]:
print(device)

cuda:1


In [31]:
convert_to_canvas(img_folder, pretrained_vit, device)



<Figure size 1000x500 with 1 Axes>

<Figure size 1000x1000 with 2 Axes>

tensor([[False, False, False,  True,  True,  True,  True,  True,  True,  True,
         False, False, False, False, False, False, False, False,  True,  True,
          True,  True, False, False, False, False, False, False, False, False,
         False, False,  True,  True,  True,  True,  True,  True, False, False,
         False, False,  True, False, False,  True,  True,  True,  True,  True,
          True,  True,  True, False, False, False,  True, False,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True, False,  True,
         False, False,  True,  True,  True,  True,  True,  True,  True,  True,
          True, False, False, False, False, False, False,  True,  True,  True,
          True,  True,  True,  True, False, False, False, False, False, False,
         False, False,  True,  True,  True,  True,  True, False, False, False,
         False, False, False, False, False, False,  True,  True,  True,  True,
          True,  True, False, False, False, False, F

<Figure size 1000x500 with 1 Axes>

<Figure size 1000x1000 with 2 Axes>

...

<Figure size 1000x1000 with 2 Axes>

tensor([[False, False, False, False, False, False,  True,  True, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False,  True, False, False,  True, False, False, False, False,
         False, False,  True, False, False,  True,  True,  True,  True,  True,
          True,  True,  True, False, False, False,  True, False,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True, False,  True,
          True, False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True, False, False, False, False, False,  True,  True,  True,
          True,  True,  True,  True, False, False, False, False, False, False,
         False, False,  True,  True,  True,  True,  True,  True, False, False,
         False, False, False, False, False,  True,  True,  True,  True,  True,
          True,  True, False, False, False, False,  

<Figure size 1000x500 with 1 Axes>

KeyboardInterrupt: 

<Figure size 1000x1000 with 2 Axes>